# Colab 실행 안내

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/DLFS_code/03_dlfs/Code.ipynb)

이 노트북은 로컬 Jupyter와 Google Colab에서 모두 실행할 수 있게 정리했습니다.
학습을 시작할 때는 바로 아래의 **공통 실행 환경 준비** 셀을 먼저 실행하세요.

- Colab에서는 저장소를 `/content/DLFS_code` 아래로 자동 clone합니다.
- 상대 경로가 깨지지 않도록 현재 노트북 폴더로 작업 디렉토리를 이동합니다.
- `lincoln` 패키지가 필요한 장은 import 경로를 자동으로 연결합니다.
- matplotlib 한글 폰트도 가능한 범위에서 자동 설정합니다.


In [ ]:
# 공통 실행 환경 준비
# 이 셀은 Colab과 로컬 Jupyter 사이의 환경 차이를 흡수한다.
# Colab에서는 먼저 저장소를 clone하고, 그 다음 notebook_env를 import해야 한다.
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/glasslego/2026-ml-study.git'
REPO_NAME = '2026-ml-study'
DLFS_SUBDIR = 'notebooks/DLFS_code'

if 'google.colab' in sys.modules:
    _bootstrap_clone_root = Path('/content') / REPO_NAME
    if not _bootstrap_clone_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(_bootstrap_clone_root)], check=True)
    _bootstrap_repo_root = _bootstrap_clone_root / DLFS_SUBDIR
else:
    _bootstrap_repo_root = None
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'lincoln').exists():
            _bootstrap_repo_root = candidate
            break
    if _bootstrap_repo_root is None:
        raise FileNotFoundError('DLFS_code 저장소 루트를 찾지 못했습니다.')

if str(_bootstrap_repo_root) not in sys.path:
    sys.path.insert(0, str(_bootstrap_repo_root))

from notebook_env import prepare_notebook_environment

REPO_ROOT = prepare_notebook_environment(
    notebook_dir='03_dlfs',
    needs_lincoln=False,
    ensure_mnist=False,
)


In [ ]:
# 이 노트북은 책 "Deep Learning from Scratch (밑바닥부터 시작하는 딥러닝)" 3장의
# 신경망 프레임워크를 numpy 만으로 직접 구현한다.
#
# 전체 구조 (책의 추상화 계층):
#   Operation         : 한 개의 수학 연산 (행렬곱, 편향 덧셈, 활성화 등)
#   Layer             : 여러 Operation 을 묶은 신경망의 한 층 (예: Dense)
#   Loss              : 예측값과 정답으로부터 손실값과 그 기울기를 계산
#   NeuralNetwork     : Layer 들을 순서대로 묶은 모델
#   Optimizer         : 파라미터 업데이트 규칙 (SGD 등)
#   Trainer           : 데이터, 모델, 옵티마이저를 묶어 학습 루프를 돌림
#
# 같은 개념을 PyTorch / TensorFlow 로 옮긴 버전이 동일 디렉토리의
# Code_pytorch.ipynb / Code_tensorflow.ipynb 에 있다.
import numpy as np
from numpy import ndarray

from typing import List

In [ ]:
def assert_same_shape(array: ndarray,
                      array_grad: ndarray):
    """두 ndarray 의 모양이 같은지 확인하는 헬퍼.

    역전파에서 "입력의 기울기는 입력과 같은 모양", "파라미터의 기울기는 파라미터와 같은 모양" 이라는
    불변식을 어디서 깨졌는지 빠르게 잡기 위한 안전장치다. 모양이 어긋나면 즉시 AssertionError 로 멈춘다.
    """
    assert array.shape == array_grad.shape, \
        '''
        두 ndarray의 모양이 같아야 하는데,
        첫 번째 ndarray의 모양은 {0}이고,
        두 번째 ndarray의 모양은 {1}이다.
        '''.format(tuple(array_grad.shape), tuple(array.shape))
    return None

# `Operation`, `ParamOperation` 클래스

In [ ]:
class Operation(object):
    """신경망 안에서 한 개의 수학 연산을 표현하는 추상 기반 클래스.

    설계 원칙:
        - forward() 는 입력을 self.input_ 에 저장한 뒤 self._output() 을 호출해 출력을 만든다.
        - backward() 는 chain rule 의 한 단계로, 출력에 대한 기울기(output_grad) 를 받아
          입력에 대한 기울기(input_grad) 를 돌려준다.
        - _output, _input_grad 는 구상 클래스마다 다르므로 NotImplementedError 로 두어
          템플릿 메서드 패턴을 강제한다.
    """
    def __init__(self):
        pass

    def forward(self, input_: ndarray):
        """순전파: 입력을 저장해 두고(역전파에 필요), _output() 결과를 반환."""
        # 역전파 시 기울기 계산에 입력값이 필요하므로 인스턴스 변수로 보존한다.
        self.input_ = input_

        self.output = self._output()

        return self.output


    def backward(self, output_grad: ndarray) -> ndarray:
        """역전파: 출력에 대한 기울기를 받아 입력에 대한 기울기를 돌려준다.

        chain rule 한 마디:  dL/d(input) = dL/d(output) * d(output)/d(input)
        여기서 dL/d(output) 가 output_grad 이고,
        d(output)/d(input) 부분을 _input_grad 안에 캡슐화한다.
        """
        # output_grad 는 self.output 과 같은 모양이어야 한다 (chain rule 차원 일치).
        assert_same_shape(self.output, output_grad)

        self.input_grad = self._input_grad(output_grad)

        # input_grad 는 self.input_ 과 같은 모양이어야 한다.
        assert_same_shape(self.input_, self.input_grad)
        return self.input_grad


    def _output(self) -> ndarray:
        """실제 출력을 만드는 함수. 구상 클래스에서 반드시 구현한다."""
        raise NotImplementedError()


    def _input_grad(self, output_grad: ndarray) -> ndarray:
        """입력에 대한 기울기를 만드는 함수. 구상 클래스에서 반드시 구현한다."""
        raise NotImplementedError()

In [ ]:
class ParamOperation(Operation):
    """학습 가능한 파라미터(가중치, 편향 등)를 가지는 Operation.

    Operation 과의 차이점은 단 하나: 역전파 시 파라미터에 대한 기울기(param_grad) 도 함께 계산한다.
    옵티마이저는 (param, param_grad) 쌍을 보고 파라미터를 갱신한다.
    """

    def __init__(self, param: ndarray) -> ndarray:
        """파라미터 텐서를 받아 self.param 으로 보관."""
        super().__init__()
        self.param = param

    def backward(self, output_grad: ndarray) -> ndarray:
        """역전파에서 input_grad 와 param_grad 를 동시에 계산한다.

        반환값은 input_grad 만이지만 param_grad 는 옵티마이저가 self.param_grad 로 꺼내서 쓴다.
        """

        assert_same_shape(self.output, output_grad)

        self.input_grad = self._input_grad(output_grad)
        self.param_grad = self._param_grad(output_grad)

        # 입력/파라미터 기울기 둘 다 모양이 어긋나면 즉시 발견되어야 한다.
        assert_same_shape(self.input_, self.input_grad)
        assert_same_shape(self.param, self.param_grad)

        return self.input_grad

    def _param_grad(self, output_grad: ndarray) -> ndarray:
        """파라미터에 대한 기울기. 구상 클래스에서 반드시 구현한다."""
        raise NotImplementedError()

## `Operation`의 구상 클래스

In [ ]:
class WeightMultiply(ParamOperation):
    """전결합층의 핵심 연산: output = input @ W

    수식 (배치 N, 입력 차원 D_in, 출력 차원 D_out):
        forward : Y      = X @ W                  ; X: (N, D_in), W: (D_in, D_out), Y: (N, D_out)
        backward: dL/dX  = dL/dY @ W^T            ; (N, D_in)
                  dL/dW  = X^T  @ dL/dY           ; (D_in, D_out)
    """

    def __init__(self, W: ndarray):
        """가중치 W 를 부모 ParamOperation 의 self.param 으로 등록."""
        super().__init__(W)

    def _output(self) -> ndarray:
        """Y = X @ W"""
        return np.dot(self.input_, self.param)

    def _input_grad(self, output_grad: ndarray) -> ndarray:
        """dL/dX = dL/dY @ W^T (chain rule)"""
        return np.dot(output_grad, np.transpose(self.param, (1, 0)))

    def _param_grad(self, output_grad: ndarray)  -> ndarray:
        """dL/dW = X^T @ dL/dY (chain rule)"""
        return np.dot(np.transpose(self.input_, (1, 0)), output_grad)

In [ ]:
class BiasAdd(ParamOperation):
    """편향을 더하는 연산: output = input + B (브로드캐스팅)

    수식 (배치 N, 출력 차원 D):
        forward : Y     = X + B            ; X: (N, D), B: (1, D), Y: (N, D)
        backward: dL/dX = dL/dY            ; X 와 같은 모양으로 그대로 흘려보냄
                  dL/dB = sum_n dL/dY[n]   ; 배치 차원에 대해 합 -> (1, D)
    """

    def __init__(self,
                 B: ndarray):
        """편향 B 의 모양은 (1, D) 여야 한다 (브로드캐스팅 + 검증 용이)."""
        # 책에서는 편향을 row vector 로 통일한다. 잘못된 모양을 미리 차단.
        assert B.shape[0] == 1

        super().__init__(B)

    def _output(self) -> ndarray:
        """Y = X + B (numpy 가 (1, D) 를 (N, D) 로 자동 확장)."""
        return self.input_ + self.param

    def _input_grad(self, output_grad: ndarray) -> ndarray:
        """덧셈의 입력 미분은 1, 그래서 그대로 흘려보낸다."""
        return np.ones_like(self.input_) * output_grad

    def _param_grad(self, output_grad: ndarray) -> ndarray:
        """편향은 모든 샘플에 같은 값이 더해지므로 배치 차원에 대해 합산한다."""
        param_grad = np.ones_like(self.param) * output_grad
        return np.sum(param_grad, axis=0).reshape(1, param_grad.shape[1])

In [ ]:
class Sigmoid(Operation):
    """시그모이드 활성화: y = 1 / (1 + exp(-x))

    역전파 공식 (출력값 y 만으로 표현 가능):
        dy/dx = y * (1 - y)
    이 성질 덕분에 forward 에서 계산한 self.output 을 그대로 재사용한다.
    """

    def __init__(self) -> None:
        super().__init__()

    def _output(self) -> ndarray:
        """y = 1 / (1 + exp(-x))"""
        return 1.0/(1.0+np.exp(-1.0 * self.input_))

    def _input_grad(self, output_grad: ndarray) -> ndarray:
        """dL/dx = dL/dy * y * (1 - y)"""
        sigmoid_backward = self.output * (1.0 - self.output)
        input_grad = sigmoid_backward * output_grad
        return input_grad

In [ ]:
class Linear(Operation):
    """항등 활성화: y = x

    회귀 문제의 마지막 층처럼 "활성화를 쓰지 않는" 경우를 표현하기 위한 편의 클래스.
    forward 와 backward 모두 입력을 그대로 통과시킨다.
    """

    def __init__(self) -> None:
        super().__init__()

    def _output(self) -> ndarray:
        """y = x"""
        return self.input_

    def _input_grad(self, output_grad: ndarray) -> ndarray:
        """dy/dx = 1, 따라서 dL/dx = dL/dy 그대로."""
        return output_grad

# `Layer`와 `Dense` 클래스

In [ ]:
class Layer(object):
    """여러 Operation 을 순서대로 묶은 신경망의 한 층.

    역할:
        - forward 는 자신이 가진 Operation 들을 순서대로 통과시키며 마지막 결과를 출력한다.
        - backward 는 같은 Operation 들을 역순으로 통과시키며 chain rule 을 자동으로 적용한다.
        - 처음 forward 호출 때 입력 모양을 보고 자식 클래스의 _setup_layer() 가 파라미터를 만든다.

    이 추상화 덕분에 NeuralNetwork 는 "Layer 의 리스트" 만 알면 학습 루프를 돌릴 수 있다.
    """

    def __init__(self,
                 neurons: int):
        """neurons 는 이 층의 출력 차원(뉴런 수)."""
        self.neurons = neurons
        self.first = True   # 첫 forward 인지 표시 (지연 초기화)
        self.params: List[ndarray] = []
        self.param_grads: List[ndarray] = []
        self.operations: List[Operation] = []

    def _setup_layer(self, num_in: int) -> None:
        """층 내부의 Operation 과 파라미터 초기화. 자식 클래스에서 구현."""
        raise NotImplementedError()

    def forward(self, input_: ndarray) -> ndarray:
        """입력을 가진 Operation 들에 순서대로 통과시킨다.

        지연 초기화: 첫 호출에서 입력 차원을 알게 되므로, 그 시점에 파라미터를 만든다.
        """
        if self.first:
            self._setup_layer(input_)
            self.first = False

        self.input_ = input_

        for operation in self.operations:

            input_ = operation.forward(input_)

        self.output = input_

        return self.output

    def backward(self, output_grad: ndarray) -> ndarray:
        """output_grad 를 Operation 역순으로 흘려보내며 입력에 대한 기울기를 만든다.

        부수효과로 각 ParamOperation 의 self.param_grad 가 채워진다.
        그 결과를 _param_grads() 로 모아서 옵티마이저가 사용할 수 있게 한다.
        """

        assert_same_shape(self.output, output_grad)

        for operation in reversed(self.operations):
            output_grad = operation.backward(output_grad)

        input_grad = output_grad

        self._param_grads()

        return input_grad

    def _param_grads(self) -> ndarray:
        """이번 backward 에서 계산된 파라미터 기울기를 self.param_grads 에 모은다."""

        self.param_grads = []
        for operation in self.operations:
            if issubclass(operation.__class__, ParamOperation):
                self.param_grads.append(operation.param_grad)

    def _params(self) -> ndarray:
        """현재 파라미터들을 self.params 에 모은다 (옵티마이저가 갱신 대상으로 사용)."""

        self.params = []
        for operation in self.operations:
            if issubclass(operation.__class__, ParamOperation):
                self.params.append(operation.param)

In [ ]:
class Dense(Layer):
    """전결합층 (Fully Connected Layer): y = activation(x @ W + b)

    내부적으로 [WeightMultiply -> BiasAdd -> activation] 세 Operation 을 순서대로 쌓은 것에 해당한다.
    이 구성 자체가 책에서 강조하는 "Layer 는 Operation 의 합성" 이라는 개념을 보여준다.
    """
    def __init__(self,
                 neurons: int,
                 activation: Operation = Sigmoid()):
        """neurons 는 출력 차원, activation 은 마지막에 적용할 활성화 함수 인스턴스."""
        super().__init__(neurons)
        self.activation = activation

    def _setup_layer(self, input_: ndarray) -> None:
        """첫 입력의 모양을 보고 W, b 를 정규분포로 초기화하고 Operation 리스트를 구성한다."""
        if self.seed:
            # NeuralNetwork 가 모든 층에 동일한 seed 를 주입해 재현성을 확보한다.
            np.random.seed(self.seed)

        self.params = []

        # 가중치 W: (input_dim, neurons)
        self.params.append(np.random.randn(input_.shape[1], self.neurons))

        # 편향 b: (1, neurons) — BiasAdd 는 row vector 만 받는다.
        self.params.append(np.random.randn(1, self.neurons))

        # Operation 들을 책의 순서 그대로 쌓는다: 행렬곱 -> 편향 -> 활성화
        self.operations = [WeightMultiply(self.params[0]),
                           BiasAdd(self.params[1]),
                           self.activation]

        return None

# `Loss`와 `MeanSquaredError` 클래스

In [ ]:
class Loss(object):
    """손실함수의 추상 기반 클래스.

    설계:
        - forward(prediction, target) -> 스칼라 손실값
        - backward()                  -> 예측값에 대한 손실의 기울기 (NeuralNetwork.backward 의 시작점)
    """

    def __init__(self):
        pass

    def forward(self, prediction: ndarray, target: ndarray) -> float:
        """예측과 정답의 모양이 같은지 확인하고 _output() 으로 실제 손실을 계산."""
        assert_same_shape(prediction, target)

        self.prediction = prediction
        self.target = target

        loss_value = self._output()

        return loss_value

    def backward(self) -> ndarray:
        """예측값에 대한 손실의 기울기 dL/d(prediction) 을 돌려준다."""
        self.input_grad = self._input_grad()

        # 예측값과 같은 모양이어야 NeuralNetwork.backward 와 연결된다.
        assert_same_shape(self.prediction, self.input_grad)

        return self.input_grad

    def _output(self) -> float:
        """실제 손실값 계산. 자식 클래스에서 구현."""
        raise NotImplementedError()

    def _input_grad(self) -> ndarray:
        """예측값에 대한 손실의 기울기. 자식 클래스에서 구현."""
        raise NotImplementedError()

In [ ]:
class MeanSquaredError(Loss):
    """회귀용 평균제곱오차 손실:

        L(y_hat, y) = (1/N) * sum_n (y_hat_n - y_n)^2

    예측에 대한 기울기:

        dL/d(y_hat) = (2/N) * (y_hat - y)

    여기서 N 은 배치 크기다 (batch dimension 의 평균을 명시적으로 잡아준다).
    """

    def __init__(self) -> None:
        super().__init__()

    def _output(self) -> float:
        """L = sum((y_hat - y)^2) / N"""
        loss = (
            np.sum(np.power(self.prediction - self.target, 2)) /
            self.prediction.shape[0]
        )

        return loss

    def _input_grad(self) -> ndarray:
        """dL/d(y_hat) = 2(y_hat - y) / N"""
        return 2.0 * (self.prediction - self.target) / self.prediction.shape[0]

# `NeuralNetwork` 클래스

In [ ]:
class NeuralNetwork(object):
    """Layer 들을 순서대로 묶은 모델.

    핵심 책임:
        - forward      : 입력 X 를 모든 Layer 에 순방향으로 통과시켜 예측을 만든다.
        - backward     : Loss.backward() 결과를 모든 Layer 에 역순으로 흘려보낸다.
        - train_batch  : 한 배치에 대해 forward -> loss -> backward 를 한 번 수행한다.
        - params/param_grads : 옵티마이저가 갱신할 파라미터/기울기를 yield 로 노출한다.
    """
    def __init__(self,
                 layers: List[Layer],
                 loss: Loss,
                 seed: int = 1) -> None:
        """층 리스트, 손실함수, 재현성용 seed 를 받는다."""
        self.layers = layers
        self.loss = loss
        self.seed = seed
        if seed:
            # 모든 층이 동일한 seed 로 가중치를 초기화하도록 시드를 주입한다.
            for layer in self.layers:
                setattr(layer, "seed", self.seed)

    def forward(self, x_batch: ndarray) -> ndarray:
        """x_batch 를 첫 층부터 끝 층까지 순서대로 통과."""
        x_out = x_batch
        for layer in self.layers:
            x_out = layer.forward(x_out)

        return x_out

    def backward(self, loss_grad: ndarray) -> None:
        """loss.backward() 결과를 받아 끝 층부터 첫 층까지 역순 전파."""

        grad = loss_grad
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

        return None

    def train_batch(self,
                    x_batch: ndarray,
                    y_batch: ndarray) -> float:
        """한 배치에 대해 [forward -> loss -> backward] 한 사이클을 수행하고 손실값을 반환.

        주의: 파라미터 갱신(step)은 여기서 하지 않는다. Trainer 가 Optimizer.step() 으로 처리.
        """

        predictions = self.forward(x_batch)

        loss = self.loss.forward(predictions, y_batch)

        # loss.backward() 가 dL/d(prediction) 을 만들고, 그것이 NeuralNetwork.backward 의 시작점.
        self.backward(self.loss.backward())

        return loss

    def params(self):
        """모든 층의 파라미터를 평탄화해서 yield. 옵티마이저가 zip 해서 사용한다."""
        for layer in self.layers:
            yield from layer.params

    def param_grads(self):
        """params() 와 같은 순서로 파라미터 기울기를 yield. 두 generator 의 순서가 같아야 한다."""
        for layer in self.layers:
            yield from layer.param_grads    

# `Optimizer`와 `SGD` 클래스

In [ ]:
class Optimizer(object):
    """파라미터 업데이트 규칙의 추상 기반 클래스.

    Trainer 가 net 을 optimizer 의 self.net 으로 주입해 주므로,
    구상 클래스의 step() 안에서 self.net.params() / self.net.param_grads() 를 zip 해 갱신한다.
    """
    def __init__(self,
                 lr: float = 0.01):
        """학습률(lr)은 모든 옵티마이저에서 공통으로 필요한 하이퍼파라미터."""
        self.lr = lr

    def step(self) -> None:
        """파라미터를 한 번 갱신. 자식 클래스에서 구현."""
        pass

In [ ]:
class SGD(Optimizer):
    """가장 단순한 확률적 경사 하강법:

        param <- param - lr * d(loss)/d(param)

    모멘텀이나 적응적 학습률은 없다. 책에서는 이 단순한 SGD 만으로 충분히 학습되는 것을 보여준다.
    """
    def __init__(self,
                 lr: float = 0.01) -> None:
        super().__init__(lr)

    def step(self):
        """한 미니배치 backward 직후에 호출되어 모든 파라미터를 한 번 갱신한다.

        주의: numpy 의 in-place 연산(-=)을 사용해 NeuralNetwork 가 들고 있는 ndarray 를 직접 수정한다.
        param = param - lr * grad 처럼 새로 할당하면 모델이 가진 참조와 끊어지므로 주의.
        """
        for (param, param_grad) in zip(self.net.params(),
                                       self.net.param_grads()):

            param -= self.lr * param_grad

# `Trainer` 클래스

In [ ]:
from copy import deepcopy
from typing import Tuple

class Trainer(object):
    """NeuralNetwork + Optimizer 를 묶어 미니배치 학습 루프를 돌린다.

    추가로 다음을 수행한다:
        - 매 epoch 마다 학습 데이터를 셔플 (permute_data)
        - eval_every 마다 테스트 데이터로 손실을 측정해 "검증 손실이 더 이상 줄지 않으면 학습 종료"
          하는 단순 early stopping 을 구현
        - 가장 좋았던 시점의 모델 가중치를 deepcopy 로 저장해 두었다가 손실이 튀면 복원
    """
    def __init__(self,
                 net: NeuralNetwork,
                 optim: Optimizer) -> None:
        """옵티마이저가 모델 파라미터를 알 수 있도록 self.optim.net 을 주입."""
        self.net = net
        self.optim = optim
        self.best_loss = 1e9
        # 옵티마이저가 step() 안에서 self.net.params() / param_grads() 를 호출할 수 있게 한다.
        setattr(self.optim, 'net', self.net)

    def generate_batches(self,
                         X: ndarray,
                         y: ndarray,
                         size: int = 32) -> Tuple[ndarray]:
        """X, y 를 size 크기의 미니배치로 잘라 yield 하는 제너레이터."""
        assert X.shape[0] == y.shape[0], \
        '''
        특징과 목푯값은 행의 수가 같아야 하는데,
        특징은 {0}행, 목푯값은 {1}행이다
        '''.format(X.shape[0], y.shape[0])

        N = X.shape[0]

        for ii in range(0, N, size):
            X_batch, y_batch = X[ii:ii+size], y[ii:ii+size]

            yield X_batch, y_batch


    def fit(self, X_train: ndarray, y_train: ndarray,
            X_test: ndarray, y_test: ndarray,
            epochs: int=100,
            eval_every: int=10,
            batch_size: int=32,
            seed: int = 1,
            restart: bool = True)-> None:
        """epochs 만큼 미니배치 학습을 수행한다.

        eval_every epoch 마다:
            1) 현재 모델을 deepcopy 로 백업
            2) 한 epoch 더 학습
            3) 테스트 손실을 측정 -> 더 좋아졌으면 best 갱신, 나빠졌으면 백업본으로 되돌리고 종료
        """

        np.random.seed(seed)
        if restart:
            # 같은 Trainer 객체로 fit 을 재호출할 때 층의 지연 초기화 플래그를 리셋한다.
            for layer in self.net.layers:
                layer.first = True

            self.best_loss = 1e9

        for e in range(epochs):

            if (e+1) % eval_every == 0:

                # eval 직전 모델을 백업 (검증 손실이 나빠지면 이쪽으로 복귀)
                last_model = deepcopy(self.net)

            # 매 epoch 마다 학습 데이터를 셔플해 배치 구성을 바꾼다.
            X_train, y_train = permute_data(X_train, y_train)

            batch_generator = self.generate_batches(X_train, y_train,
                                                    batch_size)

            for ii, (X_batch, y_batch) in enumerate(batch_generator):

                # 한 배치 forward+loss+backward
                self.net.train_batch(X_batch, y_batch)

                # 누적된 기울기로 한 스텝 갱신
                self.optim.step()

            if (e+1) % eval_every == 0:

                test_preds = self.net.forward(X_test)
                loss = self.net.loss.forward(test_preds, y_test)

                if loss < self.best_loss:
                    print(f"{e+1} 에폭에서 검증 데이터에 대한 손실값: {loss:.3f}")
                    self.best_loss = loss
                else:
                    # 손실이 증가하면 직전 백업으로 복원하고 학습 종료 (간단한 early stopping)
                    print(f"""{e+1}에폭에서 손실값이 증가했다. 마지막으로 측정한 손실값은 {e+1-eval_every}에폭까지 학습된 모델에서 계산된 {self.best_loss:.3f}이다.""")
                    self.net = last_model
                    # 옵티마이저가 새로 복원된 net 을 가리키도록 다시 주입
                    setattr(self.optim, 'net', self.net)
                    break

#### 평가 기준

In [ ]:
def mae(y_true: ndarray, y_pred: ndarray):
    """평균절대오차(MAE) = mean(|y_true - y_pred|)."""
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true: ndarray, y_pred: ndarray):
    """제곱근 평균제곱오차(RMSE) = sqrt(mean((y_true - y_pred)^2))."""
    return np.sqrt(np.mean(np.power(y_true - y_pred, 2)))

def eval_regression_model(model: NeuralNetwork,
                          X_test: ndarray,
                          y_test: ndarray):
    """테스트 셋에 대해 MAE / RMSE 를 출력한다 (회귀 평가용 헬퍼)."""
    preds = model.forward(X_test)
    preds = preds.reshape(-1, 1)
    print("평균절대오차: {:.2f}".format(mae(preds, y_test)))
    print()
    print("제곱근 평균제곱오차 {:.2f}".format(rmse(preds, y_test)))

In [ ]:
# 책에서는 동일한 데이터(보스턴 주택 가격)에 세 가지 모델을 학습해 비교한다.
#
#   lr  : 활성화 없는 1층 = 선형회귀
#   nn  : 은닉층 1개(13뉴런 + sigmoid) -> 출력층(linear)
#   dl  : 은닉층 2개(각 13뉴런 + sigmoid) -> 출력층(linear) = "딥" 모델
#
# seed 를 동일하게 줘서 실행마다 같은 결과가 나오게 한다.
lr = NeuralNetwork(
    layers=[Dense(neurons=1,
                   activation=Linear())],
    loss=MeanSquaredError(),
    seed=20190501
)

nn = NeuralNetwork(
    layers=[Dense(neurons=13,
                   activation=Sigmoid()),
            Dense(neurons=1,
                   activation=Linear())],
    loss=MeanSquaredError(),
    seed=20190501
)

dl = NeuralNetwork(
    layers=[Dense(neurons=13,
                   activation=Sigmoid()),
            Dense(neurons=13,
                   activation=Sigmoid()),
            Dense(neurons=1,
                   activation=Linear())],
    loss=MeanSquaredError(),
    seed=20190501
)

### 데이터 로드, 테스트 / 학습 데이터 분할

In [ ]:
# 보스턴 주택 가격 데이터셋.
# 주의: scikit-learn 1.2 부터 load_boston 은 윤리적 이슈로 제거되었다.
#       구버전 sklearn(1.0/1.1) 또는 fetch_openml('boston', version=1) 우회가 필요할 수 있다.
#       동일 디렉토리의 PyTorch / TensorFlow 노트북은 fetch_california_housing 을 대신 사용한다.
from sklearn.datasets import load_boston

boston = load_boston()
data = boston.data
target = boston.target
features = boston.feature_names

In [ ]:
# 특성 스케일링: 평균 0, 분산 1 로 표준화.
# 시그모이드 활성화는 입력의 절댓값이 너무 크면 포화(gradient ~ 0)되므로 표준화가 학습에 결정적이다.
from sklearn.preprocessing import StandardScaler
s = StandardScaler()
data = s.fit_transform(data)

In [ ]:
def to_2d_np(a: ndarray,
          type: str="col") -> ndarray:
    """1차원 벡터를 2차원으로 변환.

    이 프레임워크의 모든 연산은 (N, D) 모양을 가정하므로,
    sklearn 이 돌려준 (N,) 짜리 target 을 (N, 1) 로 바꿔 줘야 한다.
    """

    assert a.ndim == 1, \
    "입력된 텐서는 1차원이어야 함"

    if type == "col":
        return a.reshape(-1, 1)
    elif type == "row":
        return a.reshape(1, -1)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.3, random_state=80718)

# target 을 (N,) -> (N, 1) 로 변환해 (N, D) 가정에 맞춘다.
y_train, y_test = to_2d_np(y_train), to_2d_np(y_test)

### 3가지 모델 학습

In [ ]:
# 매 epoch 시작 시 X, y 를 같은 순열로 섞는 헬퍼.
# 미니배치가 매번 같은 순서로 들어가지 않도록 SGD 의 "확률적" 성질을 보장한다.

def permute_data(X, y):
    perm = np.random.permutation(X.shape[0])
    return X[perm], y[perm]

In [ ]:
# 모델 1: 선형회귀 — 비선형 활성화가 전혀 없으므로 가장 단순한 baseline.
trainer = Trainer(lr, SGD(lr=0.01))

trainer.fit(X_train, y_train, X_test, y_test,
       epochs = 50,
       eval_every = 10,
       seed=20190501);
print()
eval_regression_model(lr, X_test, y_test)

In [ ]:
# 모델 2: 은닉층 1개 + sigmoid — 비선형 표현력 추가. 검증 손실이 lr 보다 확실히 낮아진다.
trainer = Trainer(nn, SGD(lr=0.01))

trainer.fit(X_train, y_train, X_test, y_test,
       epochs = 50,
       eval_every = 10,
       seed=20190501);
print()
eval_regression_model(nn, X_test, y_test)

In [ ]:
# 모델 3: 은닉층 2개 + sigmoid — "딥" 모델.
# 동일 epoch 에서 nn 보다 더 낮은 검증 손실을 낸다 (단, 데이터/하이퍼에 따라 더 오래 학습이 필요할 수도 있음).
trainer = Trainer(dl, SGD(lr=0.01))

trainer.fit(X_train, y_train, X_test, y_test,
       epochs = 50,
       eval_every = 10,
       seed=20190501);
print()
eval_regression_model(dl, X_test, y_test)